### Problem Statement

"Trips & Travel.Com" company wants to enable and establish a viable business model to expand
the customer base. One of the ways to expand the customer base is to introduce a new offering of
packages. Currently, there are 5 types of packages the company is offering * Basic, Standard,
Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the
customers purchased the packages. However, the marketing cost was quite high because
customers were contacted at random without looking at the available information. The company is
now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined
as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support
or increase one's sense of well-being. However, this time company wants to harness the available
data of existing and potential customers to make the marketing expenditure more efficient.

## Data Collection

The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-
purchase-prediction The data consists of 20 column and 4888 rows.

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
sns.set()

In [2]:
df = pd.read_csv('Travel.csv')
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


## Data Cleaning
1.Handling missing values
2.Handling Duplicates
3.check data types
4.understand datasets

In [3]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [4]:
df['Gender'].value_counts()  #combine female and fe male

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [5]:
df['MaritalStatus'].value_counts()   #combine single and unmarried

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [6]:
df['TypeofContact'].value_counts()

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64

In [7]:
#replace values
df['Gender'] = df['Gender'].replace('Fe Male','Female')
df['MaritalStatus'] = df['MaritalStatus'].replace('Single','Unmarried')

In [8]:
#checking if replacement occurs
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [9]:
#check missing values
## These features are with Nan value
features_with_na = [features for features in df.columns if df[features].isnull().sum()>1]
for feature in features_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5), '% missing values')

Age 4.62357 % missing values
TypeofContact 0.51146 % missing values
DurationOfPitch 5.13502 % missing values
NumberOfFollowups 0.92062 % missing values
PreferredPropertyStar 0.53191 % missing values
NumberOfTrips 2.86416 % missing values
NumberOfChildrenVisiting 1.35025 % missing values
MonthlyIncome 4.76678 % missing values


In [10]:
#statastics on numerical columns
df[features_with_na].select_dtypes(exclude = 'object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


## imputing null values
1. impute median for age column 
2. impute mode for typeofcontract  ----------for categorical feature
3. impute median for duration of pitch ---------------for integer/float values 
4. impute mode for Numberoffollowup as it is descrete feature
5. impute mode for preferedpropertystar
6. impute median for Numberof Trips
7. impute mode for NumberOfChildrenVisiting
8. impute median for MonthlyIncome

In [11]:
#Replacing NaN/Null Values
#Age
df.Age.fillna(df.Age.median(),inplace=True)

#TypeOfContact
df.TypeofContact.fillna(df.TypeofContact.mode()[0],inplace=True)

#DurationOfPitch
df.DurationOfPitch.fillna(df.DurationOfPitch.median(),inplace=True)

#NumberOfFollowups
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0],inplace=True)

#PreferredPropertyStar
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0],inplace=True)

#NumberOfTrips
df.NumberOfTrips.fillna(df.NumberOfTrips.median(),inplace=True)

#NumberOfChildrenVisiting
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0],inplace=True)

#MonthlyIncome
df.MonthlyIncome.fillna(df.MonthlyIncome.median(),inplace=True)

In [12]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [13]:
df.drop('CustomerID',inplace=True,axis=1)

## Feature Engineering
### Feature Extraction

In [14]:
#create new column by combining column
df['TotalVisiting'] = df['NumberOfPersonVisiting'] + df['NumberOfChildrenVisiting']
df.drop(['NumberOfPersonVisiting','NumberOfChildrenVisiting'], inplace=True,axis=1)

In [15]:
#get all the numeric features
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features:', len(num_features))

Num of Numerical Features: 12


In [16]:
#Categorial features
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of categorial Features:', len(cat_features))

Num of categorial Features: 6


In [17]:
#Discrete Feature ----> feature which does not repeated anywhere comes only once
descr_features = [feature for feature in num_features if len(df[feature].unique())<=25] 
print('Num of Descrete feature: ',len(descr_features))

Num of Descrete feature:  9


In [18]:
#continous feature
conti_features = [feature for feature in num_features if feature not in descr_features]
print('Num of Continous feature:',len(conti_features))

Num of Continous feature: 3


## Train Test split and Model Training

In [19]:
from sklearn.model_selection import train_test_split
X = df.drop('ProdTaken',axis=1)
y = df['ProdTaken']

In [20]:
y.value_counts()

ProdTaken
0    3968
1     920
Name: count, dtype: int64

In [21]:
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=.2)
X_train.shape , X_test.shape

((3910, 17), (978, 17))

In [22]:
#create column transformer with 3 types of transformer
categorial_features = X.select_dtypes(include='object').columns
numerical_features = X.select_dtypes(exclude='object').columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder',oh_transformer,categorial_features),
        ('StandardScaler',num_transformer,numerical_features)
])

In [23]:
#applying transformation in training dataset
X_train = preprocessor.fit_transform(X_train)

In [24]:
X_test = preprocessor.transform(X_test)

In [25]:
pd.DataFrame(X_train)

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.721400,-1.020350,1.284279,-0.725271,-0.127737,-0.632399,0.679690,0.782966,-0.382245,-0.774151
1,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,-0.721400,0.690023,0.282777,-0.725271,1.511598,-0.632399,0.679690,0.782966,-0.459799,0.643615
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.721400,-1.020350,0.282777,1.771041,0.418708,-0.632399,0.679690,0.782966,-0.245196,-0.065268
3,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,-0.721400,-1.020350,1.284279,-0.725271,-0.127737,-0.632399,1.408395,-1.277194,0.213475,-0.065268
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.721400,2.400396,-1.720227,-0.725271,1.511598,-0.632399,-0.049015,-1.277194,-0.024889,2.061382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3905,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.721400,-0.653841,1.284279,-0.725271,-0.674182,-0.632399,-1.506426,0.782966,-0.536973,0.643615
3906,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.455047,-0.898180,-0.718725,1.771041,-1.220627,-0.632399,1.408395,0.782966,1.529609,-0.065268
3907,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.455047,1.545210,0.282777,-0.725271,2.058043,-0.632399,-0.777720,0.782966,-0.360576,0.643615
3908,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.455047,1.789549,1.284279,-0.725271,-0.127737,-0.632399,-1.506426,0.782966,-0.252799,0.643615


## GradientBoostingClassifier Model training

In [26]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score,classification_report,precision_score, recall_score, f1_score, roc_auc_score,roc_curve

In [27]:
models ={
    'Adaboost':AdaBoostClassifier(),
    'GradientBoost': GradientBoostingClassifier()
}

for i in range(len(list(models))):
    model = list(models.values()) [i]
    model.fit(X_train, y_train)  #train model

    #make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #training set Performance
    model_train_accuracy = accuracy_score(y_train,y_train_pred)
    model_train_f1 = f1_score(y_train,y_train_pred,average='weighted')
    model_train_precision = precision_score(y_train,y_train_pred)
    model_train_recall = recall_score(y_train,y_train_pred)
    model_train_rocauc_score = roc_auc_score(y_train,y_train_pred)

    #test set performance
    model_test_accuracy = accuracy_score(y_test,y_test_pred)
    model_test_f1 = f1_score(y_test,y_test_pred,average='weighted')
    model_test_precision = precision_score(y_test,y_test_pred)
    model_test_recall = recall_score(y_test,y_test_pred)
    model_test_rocauc_score = roc_auc_score(y_test,y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance for training set')
    print('Accuracy: {:.4f}'.format(model_train_accuracy))
    print('F1 Score: {:.4f}'.format(model_train_f1))
    print('precision: {:.4f}'.format(model_train_precision))
    print('Recall: {:.4f}'.format(model_train_recall))
    print('Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    print('--------------------------------------------')

    print('Model Performance for testing set')
    print('Accuracy: {:.4f}'.format(model_test_accuracy))
    print('F1 Score: {:.4f}'.format(model_test_f1))
    print('precision: {:.4f}'.format(model_test_precision))
    print('Recall: {:.4f}'.format(model_test_recall))
    print('Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    print('='*35)
    print('\n')
    

Adaboost
Model Performance for training set
Accuracy: 0.8478
F1 Score: 0.8146
precision: 0.7815
Recall: 0.2551
Roc Auc Score: 0.6194
--------------------------------------------
Model Performance for testing set
Accuracy: 0.8354
F1 Score: 0.7987
precision: 0.7500
Recall: 0.2356
Roc Auc Score: 0.6083


GradientBoost
Model Performance for training set
Accuracy: 0.8939
F1 Score: 0.8819
precision: 0.8756
Recall: 0.5021
Roc Auc Score: 0.7429
--------------------------------------------
Model Performance for testing set
Accuracy: 0.8589
F1 Score: 0.8398
precision: 0.7732
Recall: 0.3927
Roc Auc Score: 0.6824




## Hyper Parameter Tunning

In [33]:
adaboost_params = {
    "n_estimators":[50,60,70,80,90],
    "algorithm":['SAMME','SAMME.R']
}
gradient_params={"loss": ['log_loss' ,'deviance' ,'exponential'],
"criterion": ['friedman_mse','squared_error' ,'mse' ],
"min_samples_split": [2, 8, 15, 20],
"n_estimators": [100, 200, 500, 1000],
"max_depth": [5, 8, 15, None, 10]
}


In [34]:
gradient_params

{'loss': ['log_loss', 'deviance', 'exponential'],
 'criterion': ['friedman_mse', 'squared_error', 'mse'],
 'min_samples_split': [2, 8, 15, 20],
 'n_estimators': [100, 200, 500, 1000],
 'max_depth': [5, 8, 15, None, 10]}

In [35]:
#model list for parameter tunning
randomcv_models = [
    ('Adaboost',AdaBoostClassifier(),adaboost_params),
    ('Gradient',GradientBoostingClassifier(),gradient_params)
]

In [36]:
from sklearn.model_selection import RandomizedSearchCV

model_param = {}

for name, model, params in randomcv_models:
    # 1. Create the RandomizedSearchCV wrapper
    # 2. Pass the model instance to 'estimator'
    # 3. Pass params to 'param_distributions'
    random_search = RandomizedSearchCV(
        estimator=model,                # The model instance created above
        param_distributions=params,     # Your dictionary of params
        n_iter=100,
        cv=5,
        verbose=2,                      # Fixed typo: verbose
        n_jobs=-1,                      # Fixed typo: n_jobs
        random_state=42                 # Good practice to set seed
    )

    # Fit the search object
    random_search.fit(X_train, y_train)

    # Store the best parameters
    model_param[name] = random_search.best_params_

for model_name in model_param:
    print(f"----------------Best Params for {model_name}---------------")
    print(model_param[model_name])
    

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 100 candidates, totalling 500 fits
----------------Best Params for Adaboost---------------
{'n_estimators': 90, 'algorithm': 'SAMME'}
----------------Best Params for Gradient---------------
{'n_estimators': 500, 'min_samples_split': 20, 'max_depth': 10, 'loss': 'log_loss', 'criterion': 'friedman_mse'}


In [37]:
models ={
    'Adaboost': AdaBoostClassifier(n_estimators = 90, algorithm ='SAMME'),
    'Gradient': GradientBoostingClassifier(n_estimators=500,min_samples_split=20,max_depth=10,loss='log_loss',criterion='friedman_mse')
}

for i in range(len(list(models))):
    model = list(models.values()) [i]
    model.fit(X_train, y_train)  #train model

    #make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #training set Performance
    model_train_accuracy = accuracy_score(y_train,y_train_pred)
    model_train_f1 = f1_score(y_train,y_train_pred,average='weighted')
    model_train_precision = precision_score(y_train,y_train_pred)
    model_train_recall = recall_score(y_train,y_train_pred)
    model_train_rocauc_score = roc_auc_score(y_train,y_train_pred)

    #test set performance
    model_test_accuracy = accuracy_score(y_test,y_test_pred)
    model_test_f1 = f1_score(y_test,y_test_pred,average='weighted')
    model_test_precision = precision_score(y_test,y_test_pred)
    model_test_recall = recall_score(y_test,y_test_pred)
    model_test_rocauc_score = roc_auc_score(y_test,y_test_pred)

    print(list(models.keys())[i])

    print('Model Performance for training set')
    print('Accuracy: {:.4f}'.format(model_train_accuracy))
    print('F1 Score: {:.4f}'.format(model_train_f1))
    print('precision: {:.4f}'.format(model_train_precision))
    print('Recall: {:.4f}'.format(model_train_recall))
    print('Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    print('--------------------------------------------')

    print('Model Performance for testing set')
    print('Accuracy: {:.4f}'.format(model_test_accuracy))
    print('F1 Score: {:.4f}'.format(model_test_f1))
    print('precision: {:.4f}'.format(model_test_precision))
    print('Recall: {:.4f}'.format(model_test_recall))
    print('Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    print('='*35)
    print('\n')

Adaboost
Model Performance for training set
Accuracy: 0.8473
F1 Score: 0.8142
precision: 0.7750
Recall: 0.2551
Roc Auc Score: 0.6191
--------------------------------------------
Model Performance for testing set
Accuracy: 0.8364
F1 Score: 0.7977
precision: 0.7818
Recall: 0.2251
Roc Auc Score: 0.6049


Gradient
Model Performance for training set
Accuracy: 1.0000
F1 Score: 1.0000
precision: 1.0000
Recall: 1.0000
Roc Auc Score: 1.0000
--------------------------------------------
Model Performance for testing set
Accuracy: 0.9601
F1 Score: 0.9588
precision: 0.9691
Recall: 0.8220
Roc Auc Score: 0.9078


